In [0]:
%uv pip install databricks-feature-engineering
import logging
from databricks.feature_engineering import FeatureEngineeringClient,FeatureLookup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import os
import pickle
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)

In [0]:
logger = logging.getLogger("MLTraining")
logger.setLevel(logging.INFO)

In [0]:
class MLModelTraining:
    def __init__(self,spark,feature_client,feature_table,output_table):
        self.spark = spark
        self.feature_client = feature_client
        self.feature_table_name = feature_table
        self.output_table_name = output_table
        logger.info("ML Model Training instantiated")

    def create_training_dataset(self):
        logger.info("Reading the %s",self.output_table_name)
        new_df = self.spark.table(self.output_table_name)
        training_df = self.feature_client.create_training_set(
            df = new_df,
            feature_lookups = [FeatureLookup(table_name=self.feature_table_name,lookup_key="primary_key")],
            label = "total_demand",
            exclude_columns = "primary_key"
        )
        logger.info("Joined the input and the target data")
        return training_df.load_df()
    
    def sort_data(self, df):
        logger.info("Sorting data by transaction_date")
        df = df.orderBy("transaction_date")
        new_df = df.drop(df["transaction_date"])
        return new_df
    
    def split_data(self,df):
        logger.info("Split data into training,validation and testing data")
        data_df = df.toPandas()
        training_df,test_df = train_test_split(data_df,test_size = 0.20,shuffle = False)
        train_df,validate_df = train_test_split(training_df,test_size = 0.20,shuffle = False)
        return train_df,validate_df,test_df
    
    def split_input_output(self, train_df, validation_df, test_df):

        logger.info("Splitting data into X and y")
        X_train = train_df.drop(columns=["total_demand"])
        y_train = train_df["total_demand"]
        X_validate = validation_df.drop(columns=["total_demand"])
        y_validate = validation_df["total_demand"]
        X_test = test_df.drop(columns=["total_demand"])
        y_test = test_df["total_demand"]
        logger.info("X and y split completed")
        logger.info("Train X: %s | Train y: %s",X_train.shape,y_train.shape)
        logger.info("Validation X: %s | Validation y: %s",X_validate.shape,y_validate.shape)
        logger.info("Test X: %s | Test y: %s",X_test.shape,y_test.shape)
        return X_train,y_train,X_validate,y_validate,X_test,y_test
    
    def encode_categorical_data(self, X_train):

        logger.info("Starting categorical encoding")
        category_columns = ["product_name","destination_city"]
        rest_columns = ["avg_unit_price","total_inventory","transaction_count","demand_1","demand_7","rolling_7_day_avg","month","day_of_week"]
        encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
        X_train_encoded = encoder.fit_transform(X_train[category_columns])
        encoded_df = pd.DataFrame(X_train_encoded,columns=encoder.get_feature_names_out(category_columns),
                index=X_train.index)
        rest_df = X_train[rest_columns]
        X_train_final = pd.concat([rest_df, encoded_df],axis=1)
        logger.info("Categorical encoding completed")
        os.makedirs("models", exist_ok=True)
        encoder_path = "models/onehot_encoder.pkl"

        with open(encoder_path, "wb") as file:
            pickle.dump(encoder, file)
        logger.info("OneHotEncoder saved successfully at %s",encoder_path)
        return X_train_final, encoder
    
    def load_encoder(self):

        logger.info("Loading OneHotEncoder")
        with open("models/onehot_encoder.pkl", "rb") as file:
            encoder = pickle.load(file)
        logger.info("OneHotEncoder loaded successfully")
        return encoder

    def encode_validation_data(self, X_validate):

        logger.info("Starting categorical encoding")
        category_columns = ["product_name","destination_city"]
        rest_columns = ["avg_unit_price","total_inventory","transaction_count","demand_1","demand_7","rolling_7_day_avg","month","day_of_week"]

        encoder = self.load_encoder()
        logger.info("OneHotEncoder fetched successfully")

        X_validate_encoded = encoder.transform(X_validate[category_columns])
        encoded_df = pd.DataFrame(X_validate_encoded,columns=encoder.get_feature_names_out(category_columns),
                                  index=X_validate.index)
        rest_df = X_validate[rest_columns]
        X_validate_final = pd.concat([rest_df,encoded_df], axis=1)

        logger.info("Validation data encoding completed")
        return X_validate_final
    
    def create_experiment(self):

        experiment_name = "/Users/rovindcosta133@gmail.com/Demand_Prediction"

        mlflow.set_experiment(experiment_name)

        logger.info("MLflow experiment created/set: %s",experiment_name)

    def train_model(self,X_train,y_train,X_validate,y_validate):

        self.create_experiment()
        logger.info("Starting Decision Tree model training")
        params = {
            "criterion": "squared_error",
            "max_depth": 10,
            "min_samples_split": 10,
            "min_samples_leaf": 5,
            "random_state": 42
        }
        model = DecisionTreeRegressor(**params)
        
        with mlflow.start_run(run_name="DecisionTree_Training"):
            mlflow.log_params(params)
            mlflow.set_tags({
                "run_name": "first run",
                "model_type": "DecisionTreeRegressor",
                "problem_type": "regression",
                "dataset": "demand_prediction",
                "target": "total_demand",
                "feature_store": "oag.gold.demand_input_features"
            })
            model.fit(X_train,y_train)
            logger.info("Decision Tree model trained successfully")
            y_pred = model.predict(X_validate)
            mae = mean_absolute_error(y_validate,y_pred)
            rmse = mean_squared_error(y_validate,y_pred) ** 0.5
            r2 = r2_score(y_validate,y_pred)
            mlflow.log_metrics({
                "validation_mae": mae,
                "validation_rmse": rmse,
                "validation_r2": r2
            })

            logger.info("Validation MAE: %.4f", mae)
            logger.info("Validation RMSE: %.4f", rmse)
            logger.info("Validation R2: %.4f", r2)
            mlflow.sklearn.log_model(
                sk_model=model,
                name="decision_tree_model"
            )
            logger.info("Model logged successfully")
        #logger.info("MLflow run completed: %s",run_id)
        return model
    
    def run(self):
        logger.info("ML pipeline started")
        df = self.create_training_dataset()
        df = self.sort_data(df)
        train_df, validate_df, test_df = self.split_data(df)
        X_train,y_train,X_validate,y_validate,X_test,y_test = self.split_input_output(train_df,validate_df,test_df)
        X_train_final, encoder = self.encode_categorical_data(X_train)
        X_validate_final = self.encode_validation_data(X_validate)
        model,run_id = self.train_model(X_train_final,y_train,X_validate_final,y_validate)
        
        logger.info("ML pipeline completed")
        return True
    

In [0]:
feature_client = FeatureEngineeringClient()
ml_obj = MLModelTraining(
    spark=spark,
    feature_client=feature_client,
    feature_table="oag.gold.demand_input_features",
    output_table="oag.gold.demand_output"
)
ml_obj.run()

In [0]:
X_train_encoded


In [0]:
validation_df

In [0]:
test_df